# Corrected Aggregation from Raw Sources — Retail (P6242.1) + Macro variables

Rebuilds the monthly modelling dataset (1986-2026) entirely from **raw source files**, with no pre-amalgamated input:

- **Retail target:** Stats SA P6242.1 Total retail trade sales, **constant 2019 prices, NSA**, growth-spliced across the 1995 -> 2019 base change.
- **Electricity:** Stats SA P4141 national production **Index (base 2019=100, NSA)** — the correct single series.
- **CPI:** **BIS** Data Portal, series `M.ZA.628` (consumer price index, 2010=100, monthly).
- **Exchange rate (USD/ZAR):** **BIS** Data Portal, series `M.ZA.ZAR.A` (monthly, PERIOD AVERAGE). Download this series and save it as `bis_dp_search_export_exchangeRateUSDZAR_average.xlsx`.
- **Consumer confidence (CCI):** **BER** FNB/BER workbook (quarterly), disaggregated to monthly by a **step function** (each reading carried forward until the next).

Target validated against the official P6242.1 (May 2026) release.

In [6]:
"""
build_retail_dataset_from_raw.py
================================
Corrected aggregation of the Stats SA P6242.1 retail trade sales target and
assembly of the monthly modelling dataset (1986-2026), reading EVERY series
directly from its raw source. No pre-amalgamated covariate file is used.

SOURCES
-------
* Retail target : Stats SA P6242.1 .xls/.xlsx vintages (Total, constant prices,
                  NSA), growth-spliced across the 1995 -> 2019 base change.
* Electricity   : Stats SA P4141 (national production index, base 2019=100, NSA).
* CPI           : BIS Data Portal export, series M.ZA.628 (CPI index 2010=100).
* Exchange rate : BIS Data Portal export, series M.ZA.ZAR.A (USD/ZAR, monthly,
                  PERIOD AVERAGE).
* CCI           : BER FNB/BER consumer-confidence workbook (quarterly), carried
                  forward to monthly by a step function.

WHY THIS SCRIPT EXISTS
----------------------
The previous pipeline read the macro covariates (CPI, CCI, exchange rate) from an
already-amalgamated CSV and only re-derived the retail target and electricity.
This version removes that dependency: each macro variable is read from its own
raw export, so the dataset is reproducible end-to-end from primary sources.

The target fix is unchanged: do NOT drop the H-columns; select ONE consistent
published series (TOTAL, CONSTANT prices, NSA); splice vintages by GROWTH RATES
(chain the pre-2002 history onto the 2019-base anchor), which removes the base
seam while preserving each era's real dynamics.

VALIDATION
----------
The target matches the official P6242.1 (May 2026) release exactly:
Dec 2025 = 138,834 ; Jan-May 2026 = 97,326 / 95,209 / 99,422 / 96,677 / 101,008
(constant 2019 prices, R million).

NOTES / CAVEATS
---------------
* Target is constant 2019 prices, NSA (keeps seasonality for the models).
  Switch PRICE_BASIS to 'cur' for the current-price (nominal) total.
* Pre-2002 total is the old "type of merchandise" classification, 2002+ the
  "type of dealer" classification (a Stats SA survey redesign). December seasonal
  amplitude differs mildly (Dec/annual-mean ~1.47 pre-2002 vs ~1.38 from 2002).
* CPI (M.ZA.628) is an index on a 2010=100 base; the modelling notebook converts
  it to year-on-year inflation. The exchange rate and electricity index are
  attached on their own monthly dates.
* Exogenous covariates only run to late 2025 / early 2026; the 2026 holdout cells
  are forward-filled (end-of-sample padding) and must NOT be treated as real 2026
  information. The holdout is target-only, forecast from the Dec 2025 origin.
  Publication lags for CPI/electricity are applied in the MODELLING notebook, not
  here, so this file holds the contemporaneous monthly series.
* CCI is QUARTERLY; it is disaggregated to monthly by a step function (each
  reading carried forward until the next), not interpolated. See
  disaggregate_quarterly_to_monthly() for why interpolation / Chow-Lin would leak
  information that is not yet observable at the month being filled.
"""

import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
PRICE_BASIS = "con"      # 'con' = constant prices ; 'cur' = current prices
START = "1986-01-01"
END = "2026-05-31"

# --- Raw macro-variable source files (read directly; no amalgamated input) --- #
CPI_FILE = "bis_dp_search_export_CPI.xlsx"                 # BIS M.ZA.628, CPI index 2010=100, monthly
FX_FILE  = "bis_dp_search_export_exchangeRateUSDZAR_average.xlsx"  # BIS M.ZA.ZAR.A, USD/ZAR, monthly (PERIOD AVERAGE)
#   Download the AVERAGE series M.ZA.ZAR.A from the BIS Data Portal and save it under this name.
#   (The end-of-period series M.ZA.ZAR.E would be used if you point this at the old file instead.)
CCI_FILE = "FNB_BER_CCI.xlsx"                              # BER FNB/BER consumer confidence, quarterly

# Electricity: P4141 production index, base 2019=100, NSA (H03 == 'ELEKIN11').
# Already on a single 2019 base across files, so no splicing is needed.
ELECTRICITY_FILES = [
    "Excel table from 1985 to 1989_Electricity.xlsx",
    "Excel table from 1990 to 1999_Electricity.xlsx",
    "Excel table from 2000_Electricity.xlsx",
]
ELECTRICITY_CODE = "ELEKIN11"

RETAIL_FILES = {
    # name: (path, engine)
    "1968_1980": ("Excel 1968 to 1980.xls", "xlrd"),
    "1981_1990": ("Excel 1981 to 1990.xls", "xlrd"),
    "1991_2000": ("Excel 1991 to 2000.xls", "xlrd"),
    "from_2001": ("Excel from 2001.xls", "xlrd"),
    "from_2002": ("Retail trade sales from 2002.xlsx", None),
}

# The four FNB/BER CCI sub-series (headline net balance + three components).
CCI_COLS = [
    "cci_net_balance",
    "cci_expected_economic_performance",
    "cci_expected_household_finances",
    "cci_time2buy_durables_net",
]

# Months between the end of a CCI reference quarter and its BER publication date.
# 0 = a reading is available from the final month of its own quarter (reproduces
# the original behaviour). Publication lags are handled in the modelling notebook,
# so this stays 0 here. VERIFY AGAINST BER RELEASE DATES before changing.
CCI_PUBLICATION_LAG_MONTHS = 0


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def _mo_to_date(col):
    """'mo011986' or 'MO011986' -> Timestamp('1986-01-01')."""
    c = str(col).upper().replace("MO", "")
    return pd.Timestamp(f"{c[2:]}-{c[:2]}-01")


def _read(path, engine):
    df = pd.read_excel(path, engine=engine) if engine else pd.read_excel(path)
    df.columns = [str(c) for c in df.columns]
    return df


def historical_total(path, engine):
    """TOTAL, constant-price, ACTUAL (NSA) row from a pre-2002 file."""
    df = _read(path, engine)
    mo = [c for c in df.columns if str(c).upper().startswith("MO")]
    up = lambda n: df[n].astype(str).str.upper()
    mask = (
        up("H15").str.contains("CONSTANT", na=False)
        & up("H16").str.contains("ACTUAL", na=False)
        & up("H05").str.fullmatch("TOTAL", na=False)
    )
    sel = df[mask]
    if len(sel) == 0:
        raise ValueError(f"No constant/actual/total row found in {path}")
    s = sel[mo].iloc[0].astype(float)
    return pd.Series(s.values, index=[_mo_to_date(c) for c in mo]).sort_index()


def modern_total(path, basis="con"):
    """Modern (2002+) Total row. H03 in {con_act, con_seas, cur_act, cur_seas}."""
    code = f"{basis}_act"  # NSA total
    df = _read(path, None)
    mo = [c for c in df.columns if str(c).upper().startswith("MO")]
    sel = df[df["H03"] == code]
    if len(sel) == 0:
        raise ValueError(f"No H03=={code} Total row in {path}")
    s = sel[mo].iloc[0].astype(float)
    return pd.Series(s.values, index=[_mo_to_date(c) for c in mo]).sort_index()


def electricity_index(files=ELECTRICITY_FILES, code=ELECTRICITY_CODE):
    """National electricity production index (base 2019=100, NSA) from P4141."""
    parts = []
    for f in files:
        df = _read(f, None)
        mo = [c for c in df.columns if str(c).upper().startswith("MO")]
        row = df[df["H03"] == code]
        if len(row) == 0:
            raise ValueError(f"No H03=={code} row in {f}")
        s = row[mo].iloc[0].astype(float)
        parts.append(pd.Series(s.values, index=[_mo_to_date(c) for c in mo]))
    s = pd.concat(parts).sort_index()
    s = s[~s.index.duplicated(keep="last")]
    s.name = "electricity"
    return s


def read_bis_series(path, name, expect_key=None):
    """Read a single BIS Data Portal time series (xlsx export) into a month-end
    Series. The 'timeseries observations' sheet carries one row per month with a
    'TIME_PERIOD:Period' date and an 'OBS_VALUE:Value' observation. If expect_key
    is given, verify the file holds exactly that series (guards against pointing
    at the wrong export, e.g. end-of-period vs average FX)."""
    df = pd.read_excel(path, sheet_name="timeseries observations", header=0)
    df.columns = [str(c).split(":")[0].strip() for c in df.columns]
    if expect_key is not None:
        keys = set(df["KEY"].dropna().astype(str).str.strip())
        if keys != {expect_key}:
            raise ValueError(f"{path}: expected only series {expect_key}, found {sorted(keys)}")
    idx = pd.to_datetime(df["TIME_PERIOD"]) + pd.offsets.MonthEnd(0)
    s = pd.Series(pd.to_numeric(df["OBS_VALUE"], errors="coerce").values, index=idx)
    s = s[~s.index.duplicated(keep="last")].sort_index()
    s.name = name
    return s


def read_ber_cci(path, sheet="CCI"):
    """Read the FNB/BER consumer-confidence workbook into a QUARTERLY DataFrame
    (quarter-end dated) with the headline net balance and its three components.
    Column order on the 'CCI' sheet: date, headline, expected economic
    performance, expected household finances, time to buy durables."""
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    raw = raw[pd.to_datetime(raw[0], errors="coerce").notna()]  # drop header/notes rows
    idx = pd.to_datetime(raw[0]) + pd.offsets.MonthEnd(0)
    out = pd.DataFrame(
        {
            "cci_net_balance": pd.to_numeric(raw[1], errors="coerce").values,
            "cci_expected_economic_performance": pd.to_numeric(raw[2], errors="coerce").values,
            "cci_expected_household_finances": pd.to_numeric(raw[3], errors="coerce").values,
            "cci_time2buy_durables_net": pd.to_numeric(raw[4], errors="coerce").values,
        },
        index=idx.values,
    )
    return out[~out.index.duplicated(keep="last")].sort_index()


def growth_splice(hist, modern, join="2002-01-01"):
    """Keep `modern` from the join date onward; back-cast the pre-join history by
    applying the historical series' own month-on-month growth to the modern
    anchor. Removes the base-year seam by construction."""
    join = pd.Timestamp(join)
    out = {d: v for d, v in modern.loc[join:].items()}
    hist_g = hist.pct_change()
    back_dates = pd.date_range(START, join - pd.offsets.MonthBegin(1), freq="MS")
    prev = join
    for d in reversed(back_dates):
        g = hist_g.get(prev, np.nan)
        out[d] = out[prev] if (pd.isna(g) or g == -1) else out[prev] / (1.0 + g)
        prev = d
    s = pd.Series(out).sort_index()
    return s[~s.index.duplicated(keep="last")]


# --------------------------------------------------------------------------- #
# 1. Build the corrected TARGET
# --------------------------------------------------------------------------- #
def build_target(basis=PRICE_BASIS):
    h81 = historical_total(*RETAIL_FILES["1981_1990"])
    h91 = historical_total(*RETAIL_FILES["1991_2000"])
    h01 = historical_total(*RETAIL_FILES["from_2001"])
    # Same 1995 base across the three historical files -> direct splice.
    hist = pd.concat(
        [h81.loc[:"1990-12"], h91.loc["1991-01":"2000-12"], h01.loc["2001-01":]]
    ).sort_index()
    hist = hist[~hist.index.duplicated(keep="last")]

    modern = modern_total(RETAIL_FILES["from_2002"][0], basis=basis)  # constant 2019
    target = growth_splice(hist, modern, join="2002-01-01").loc[START:END]
    target.name = "retail_sales"
    return target


# --------------------------------------------------------------------------- #
# 1b. Quarterly -> monthly disaggregation of the CCI block (step function)
# --------------------------------------------------------------------------- #
def disaggregate_quarterly_to_monthly(cov, cci_cols, lag_months=0):
    """Spread quarterly CCI readings across monthly observations (step function).

    Each quarterly reading is carried forward to the months that follow it until
    the next reading arrives. Interpolation, splines and regression-based
    temporal disaggregation (Chow-Lin / Fernandez / Litterman) were considered
    and rejected: each one uses readings that are not yet observable at the month
    being filled, which would leak future information into a predictor. The step
    function is the only treatment that uses nothing beyond the most recently
    published reading. Months before the first reading are left NaN.
    """
    out = cov.copy()
    present = [c for c in cci_cols if c in out.columns]
    if not present:
        return out
    observed = out[present].dropna(how="all")
    if observed.empty:
        return out
    if lag_months:
        observed = observed.copy()
        observed.index = observed.index + pd.offsets.MonthEnd(lag_months)
    carried = observed.reindex(out.index.union(observed.index)).ffill()
    out[present] = carried.reindex(out.index)
    return out


def audit_cci(df, cci_cols=None, n=18):
    """Sanity check: CCI should be a step function changing four times a year."""
    cci_cols = cci_cols or CCI_COLS
    cols = [c for c in cci_cols if c in df.columns] + ["cci_level"]
    print("CCI tail (step function expected, one change per quarter):")
    print(df[cols].tail(n).to_string())
    changed = df["cci_level"].diff().ne(0) & df["cci_level"].notna()
    print("\nChanges in cci_level per year (4 expected once readings begin):")
    print(changed.groupby(df.index.year).sum().tail(12).to_string())


# --------------------------------------------------------------------------- #
# 2. Assemble the modelling dataset (target + raw covariates + derived features)
# --------------------------------------------------------------------------- #
def build_dataset():
    target = build_target()
    target.index = target.index + pd.offsets.MonthEnd(0)  # month-end to match covariates

    df = pd.DataFrame(index=target.index)
    df["retail_sales"] = target

    # --- raw macro variables, each read from its own source --------------- #
    df["cpi"] = read_bis_series(CPI_FILE, "cpi", expect_key="M.ZA.628").reindex(df.index)           # BIS M.ZA.628
    df["exchangeRate"] = read_bis_series(FX_FILE, "exchangeRate", expect_key="BIS:WS_XRU(1.0):M.ZA.ZAR.A").reindex(df.index)  # BIS M.ZA.ZAR.A

    elec = electricity_index()                                                # P4141 index (2019=100, NSA)
    elec.index = elec.index + pd.offsets.MonthEnd(0)
    df["electricity"] = elec.reindex(df.index)

    # CCI: quarterly BER readings -> monthly step function. Run the step function
    # over the full quarterly history (so the reading in force at START carries in),
    # then align to the target's monthly index.
    cci_q = read_ber_cci(CCI_FILE)
    cci_frame = pd.DataFrame(index=cci_q.index.union(df.index))
    for c in CCI_COLS:
        cci_frame[c] = cci_q[c].reindex(cci_frame.index)
    cci_frame = disaggregate_quarterly_to_monthly(cci_frame, CCI_COLS, CCI_PUBLICATION_LAG_MONTHS)
    for c in CCI_COLS:
        df[c] = cci_frame[c].reindex(df.index)

    # End-of-sample padding so covariate columns are non-empty over the holdout.
    # These frozen 2026 cells are NOT real information (holdout is target-only,
    # forecast from the Dec 2025 origin). Publication lags are applied later, in
    # the modelling notebook.
    cov_cols = ["cpi", "electricity", "exchangeRate"] + CCI_COLS
    df[cov_cols] = df[cov_cols].ffill()

    # Derived features recomputed from the CORRECTED target.
    df["retail_lag_1"] = df["retail_sales"].shift(1)
    df["retail_lag_3"] = df["retail_sales"].shift(3)
    df["retail_lag_6"] = df["retail_sales"].shift(6)
    df["retail_growth"] = df["retail_sales"].pct_change()
    df["electricity_growth"] = df["electricity"].pct_change()
    df["cci_level"] = df[CCI_COLS].mean(axis=1)

    schema = [
        "retail_sales", "cpi", "electricity", "cci_net_balance",
        "cci_expected_economic_performance", "cci_expected_household_finances",
        "cci_time2buy_durables_net", "exchangeRate", "retail_lag_1", "retail_lag_3",
        "retail_lag_6", "retail_growth", "electricity_growth", "cci_level",
    ]
    df = df[schema]
    df.index.name = "date"
    return df


# --------------------------------------------------------------------------- #
# 3. Validate against the official release
# --------------------------------------------------------------------------- #
def validate(df):
    checks = {
        "2025-12-31": 138834, "2026-01-31": 97326, "2026-02-28": 95209,
        "2026-03-31": 99422, "2026-04-30": 96677, "2026-05-31": 101008,
    }
    ok = True
    for d, v in checks.items():
        got = float(df.loc[d, "retail_sales"])
        flag = "OK" if abs(got - v) < 1 else "MISMATCH"
        ok = ok and flag == "OK"
        print(f"  {d}: ours={got:,.0f}  release={v:,d}  [{flag}]")
    print(f"  -> release validation {'PASSED' if ok else 'FAILED'}")
    return ok

## Build, validate and save

In [7]:
data = build_dataset()
print(data.shape)
validate(data)
out = data.reset_index()
out['date'] = out['date'].dt.strftime('%Y/%m/%d')
out.to_csv('final_model_dataset_corrected.csv', sep=';', index=False)

# CCI frequency-conversion check (step function, 4 changes per year)
audit_cci(data)
data[['retail_sales', 'cpi', 'electricity', 'exchangeRate', 'cci_level']].describe()

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
(485, 14)
  2025-12-31: ours=138,834  release=138,834  [OK]
  2026-01-31: ours=97,326  release=97,326  [OK]
  2026-02-28: ours=95,209  release=95,209  [OK]
  2026-03-31: ours=99,422  release=99,422  [OK]
  2026-04-30: ours=96,677  release=96,677  [OK]
  2026-05-31: ours=101,008  release=101,008  [OK]
  -> release validation PASSED
CCI tail (step function expected, one change per quarter):
            cci_net_balance  cci_expected_economic_performance  cci_expected_household_finances  cci_time2buy_durables_net  cci_level
date                                                                                                                                 
2024-12-31        -6.220493                          -8.803612                        10.875332                 -20.73319

,retail_sales,cpi,electricity,exchangeRate,cci_level
count,485.000000,485.000000,485.000000,485.000000,485.000000
mean,65526.390239,91.224478,87.438557,8.569512,-1.721399
std,24487.743945,57.911677,15.034974,5.060218,11.631193
min,33036.141023,12.988100,50.900000,1.952487,-32.486236
25%,41910.169530,42.225400,75.700000,3.751863,-9.250000
50%,65498.000000,73.768100,91.400000,7.395607,-2.750000
75%,87041.000000,137.480900,99.600000,13.149754,5.750000
max,138834.000000,211.728700,113.000000,19.066816,26.000000
